# NeoHuman — NVIDIA StyleGAN2-ADA, 256 × 256

Separate local experiment using all 45 images. **Prepared only: training is OFF.**
Wait for the existing 24-hour R3GAN to finish. No scheduled or automatic launch.

[NVIDIA paper](https://arxiv.org/abs/2006.06676) · [Official code](https://github.com/NVlabs/stylegan2-ada-pytorch)

The 8 GB laptop profile uses four-image microbatches, accumulation, mixed precision,
TF32/channels-last acceleration, and reference operations. Both training and
generation cap PyTorch-managed allocations at 85% of VRAM or free VRAM minus
1 GiB at startup, whichever is smaller. This excludes driver/other-app allocations;
exceeding the cap stops the run with an error. GPU fit is not yet verified. Native 256px output and
good results from 45 inputs are different questions; detail/diversity are not guaranteed.
See `README.md` and `VALIDATION.md` in this folder.


In [ ]:
from pathlib import Path
import json, subprocess, sys

HERE = Path.cwd().resolve()
PROJECT = HERE if (HERE / 'manage.py').exists() and (HERE / 'upstream-lock.json').exists() else HERE / 'stylegan2_ada'
assert (PROJECT / 'manage.py').exists(), 'Open the notebook from Mind_Progeny or its stylegan2_ada folder.'
PYTHON = PROJECT / '.venv' / 'Scripts' / 'python.exe'
PROFILE = '256'
assert PYTHON.exists(), 'Run Setup.ps1 in the stylegan2_ada folder first.'

def command(*args):
    # Each command uses the isolated environment, regardless of notebook kernel.
    with subprocess.Popen([str(PYTHON), '-u', str(PROJECT / 'manage.py'), '--profile', PROFILE, *args], cwd=PROJECT,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
        except KeyboardInterrupt:
            if args and args[0] == 'train':
                subprocess.run([str(PYTHON), str(PROJECT / 'manage.py'), 'stop'], cwd=PROJECT, check=True)
                print('Stop requested. Waiting for the next maintenance tick and snapshot...', flush=True)
                for line in process.stdout:
                    print(line, end='', flush=True)
            else:
                raise
        result = process.wait()
    if result:
        raise RuntimeError(f'Command stopped with exit code {result}; read its message above.')

print('Separate project:', PROJECT)
print('Separate Python:', PYTHON)


## 1. Check readiness (no GPU or model execution)

In [ ]:
command('status')
command('plan')


## 2. Inspect the 45 prepared inputs

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(PROJECT / 'data' / 'contact-sheet-256.jpg')))
manifest = json.loads((PROJECT / 'data' / 'neohuman45-256.json').read_text())
print('Images:', manifest['count'], '| resolution:', manifest['resolution'], '| enlarged inputs:', manifest['upscaled_count'])


## 3. Review the NEW run's budget

`total_kimg = 100` means 100,000 image presentations (12,500 batch updates at batch 8).
It is **not 100 hours or the previous model's 24-hour limit**. This is an initial
experiment budget, not a quality guarantee. If changing it before a run, edit
`config-256.json` in this folder and rerun the plan cell. Review previews while training.


In [ ]:
print((PROJECT / 'config-256.json').read_text())


## 4. Start manually AFTER the original 24-hour run finishes

Change `START_TRAINING` to `True` only when ready to run this new model. The launcher
still checks the original run's completion. The cell remains busy during training.
If GPU memory is insufficient, it exits; no automatic lower-resolution fallback.
Keep the charger connected and sleep set to Never.

To stop while this cell is busy, double-click `Stop-StyleGAN2-ADA.cmd` in this folder.
Keep the notebook/kernel/window open until the run saves and exits. The stop is
checked at the next maintenance tick (roughly every 1,000 image presentations).
Do not use kernel shutdown as a save command.


In [ ]:
START_TRAINING = False
RESUME_SNAPSHOT = None  # Optional absolute path to a verified snapshot from THIS project.
if START_TRAINING:
    args = ['train', '--execute']
    if RESUME_SNAPSHOT:
        args += ['--resume', str(Path(RESUME_SNAPSHOT).resolve())]
    command(*args)
else:
    print('Training stays OFF. Change START_TRAINING only when you are ready.')


## 5. Inspect saved previews and snapshots

Network snapshots restore weights only. A later `--resume` starts fresh optimizers,
ADA adaptation and counters in a separate run folder; the budget applies again.
Choose snapshots by visual quality, not merely by the highest step number.


In [ ]:
runs = sorted((PROJECT / 'runs').glob('*')) if (PROJECT / 'runs').exists() else []
for run in runs:
    print(run)
    verified = sorted(run.glob('network-snapshot-*.verified.json'))
    print('  Verified snapshots:', len(verified))
    if verified:
        print('  Latest:', str(verified[-1]).replace('.verified.json', '.pkl'))
    previews = sorted(run.glob('fakes*.png'))
    if previews:
        display(Image(filename=str(previews[-1]), width=640))
if not runs:
    print('No new training run has started yet.')


## 6. Generate native 256px PNGs after training has stopped

Set `SNAPSHOT` to the full `.pkl` path you selected above. Change `GENERATE` to
`True` only when ready. This loads the new model on the GPU and generates one image
at a time; it cannot run alongside this project's training. Nothing generates on
Run All with the defaults. Results go to a new `generated/` subfolder.


In [ ]:
GENERATE = False
SNAPSHOT = None  # Paste the full verified .pkl path here as a quoted string.
SEEDS = '0,1,2,3'
if GENERATE:
    assert SNAPSHOT, 'Choose a verified snapshot first.'
    command('generate', '--execute', '--snapshot', str(Path(SNAPSHOT).resolve()), '--seeds', SEEDS, '--truncation', '0.7')
else:
    print('Generation stays OFF.')
